# 06 · Launch HuggingFace Accelerate — Single Node

학습 로직은 [`torch_distributor_trainer.py`](./torch_distributor_trainer.py)에 있습니다. 본 노트북은 driver 역할로 `accelerate launch`를 subprocess로 호출하고 결과를 확인합니다.

| 섹션 | launcher 호출 |
|------|--------------|
| 1×1  | `accelerate launch --num_processes 1 torch_distributor_trainer.py ...` |
| 1×N  | `accelerate launch --multi_gpu --num_processes N torch_distributor_trainer.py ...` |

`torch_distributor_trainer.py`의 `__main__` 블록이 argparse로 인자를 받고 같은 `train_fn`을 호출합니다. `accelerate launch`도 `RANK` / `WORLD_SIZE` / `LOCAL_RANK` / `MASTER_ADDR` / `MASTER_PORT`를 child에 세팅하므로 `train_fn` 내부의 `dist.init_process_group("nccl")`이 그대로 동작합니다.

> Multi-node(M×N)는 native `accelerate launch --multi_gpu --num_machines M --machine_rank R`이 노드별 dispatch를 요구하지만 Databricks에서는 단일 driver 노트북에서 이를 직접 띄울 수 없습니다. 실용적인 대안은 [`07-launch_accelerator_multi_node.ipynb`](07-launch_accelerator_multi_node.ipynb) — TorchDistributor를 dispatcher로 쓰고 child에서 `Accelerator()` API를 사용.

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0, Autoscaling off. `accelerate`는 DBR ML에 사전 설치되어 있습니다 (`accelerate==1.5.2`).

In [ ]:
%run ./00-setup

## 경로 + 스크립트 위치

`accelerate launch`는 file path 기반이므로 노트북이 위치한 Workspace 경로를 절대 경로로 잡아둡니다.

In [ ]:
import os

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
TRAINER_SCRIPT = os.path.join(SCRIPT_DIR, "torch_distributor_trainer.py")
assert os.path.exists(TRAINER_SCRIPT), TRAINER_SCRIPT
print(f"SCRIPT_DIR={SCRIPT_DIR}")
print(f"TRAINER_SCRIPT={TRAINER_SCRIPT}")

## 공용 launch 헬퍼

driver에서 MLflow run을 만든 뒤 `run_id`와 모든 하이퍼파라미터를 `accelerate launch`에 전달합니다. `db_token`이 argv로 노출되긴 하지만 단일 사용자 노트북 환경에서는 허용. 더 엄격한 격리가 필요하면 `os.environ`으로 전달하도록 trainer를 수정하세요.

In [ ]:
import subprocess

import mlflow


def run_accelerate(num_processes, topology, ckpt_name):
    cfg = CONFIG
    mlflow.set_experiment(EXPERIMENT_PATH)
    with mlflow.start_run(run_name=f"acc-{topology}", log_system_metrics=True) as run:
        run_id = run.info.run_id

    cmd = [
        "accelerate", "launch",
        "--num_processes", str(num_processes),
    ]
    if num_processes > 1:
        cmd.append("--multi_gpu")
    cmd += [
        TRAINER_SCRIPT,
        "--run_id", run_id,
        "--db_host", DB_HOST,
        "--db_token", DB_TOKEN,
        "--data_dir", DATA_DIR,
        "--ckpt_path", os.path.join(CKPT_DIR, ckpt_name),
        "--n_users", str(cfg["n_users"]),
        "--n_items", str(cfg["n_items"]),
        "--emb_dim", str(cfg["emb_dim"]),
        "--tower_hidden", *[str(x) for x in cfg["tower_hidden"]],
        "--batch_size", str(cfg["batch_size_per_gpu"]),
        "--num_epochs", str(cfg["num_epochs"]),
        "--max_steps_per_epoch", str(cfg["max_steps_per_epoch"]),
        "--patience", str(PATIENCE),
        "--min_delta", str(MIN_DELTA),
        "--topology", topology,
        "--script_dir", SCRIPT_DIR,
    ]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=SCRIPT_DIR)

## 1×1 GPU

`accelerate launch --num_processes 1 torch_distributor_trainer.py ...`. `--multi_gpu`는 생략 — single-process라 DDP all_reduce는 no-op.

In [ ]:
run_accelerate(num_processes=1, topology="1x1", ckpt_name="acc-1x1.pt")

## 1×N GPU

같은 스크립트를 `--num_processes N --multi_gpu`로 호출. global batch는 `batch_size_per_gpu × N`.

In [ ]:
NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수에 맞춰 조정
run_accelerate(num_processes=NUM_GPUS_PER_NODE, topology="1xN", ckpt_name="acc-1xN.pt")